# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [1]:
import io
import re
import sys
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
from playwright.sync_api import TimeoutError as PWTimeout
from playwright.sync_api import sync_playwright

In [2]:
URL_PAGINA = "https://amefibra.com/el-mercado/indice-fibras/"
PATRON_IFRAME_TABLA = re.compile(r"edimex\.com\.mx/Emisora/Reportes/?(\?.*)?$")
CARPETA_SALIDA = Path.cwd() / "output"
FUENTE_DATOS = "AMEFIBRA / Economatica México"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Funciones de extracción

In [3]:
def _limpiar_encabezado(texto: str) -> str:
    texto = re.sub(r"[↑↓]", "", str(texto))
    return re.sub(r"\s+", " ", texto).strip()


def _localizar_frame_tabla(page, intentos=10, espera_ms=1000):
    for _ in range(intentos):
        for frame in page.frames:
            if PATRON_IFRAME_TABLA.search(frame.url or ""):
                return frame
        page.wait_for_timeout(espera_ms)
    return None


def _extraer_html_tabla(frame) -> Optional[str]:
    return frame.evaluate("""() => {
        const tablas = Array.from(document.querySelectorAll('table'));
        let mejor = null, filasMax = -1;
        for (const tabla of tablas) {
            const filas = tabla.querySelectorAll('tbody tr').length;
            if (filas > filasMax) { filasMax = filas; mejor = tabla; }
        }
        return mejor ? mejor.outerHTML : null;
    }""")


def obtener_tabla_fibras(headless: bool = True, timeout_datos_ms: int = 30000) -> pd.DataFrame:
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=headless)
        page = browser.new_page(locale="es-MX")
        try:
            page.goto(URL_PAGINA, wait_until="domcontentloaded")
            frame = _localizar_frame_tabla(page)
            if frame is None:
                raise RuntimeError("No se encontró el iframe con la tabla de FIBRAs.")
            try:
                frame.wait_for_function("""() => {
                    const filas = document.querySelectorAll('table tbody tr');
                    if (filas.length === 0) return false;
                    const celda = filas[0].querySelector('td:nth-child(2)');
                    const texto = celda ? celda.textContent.trim() : '';
                    return texto.length > 0 && texto !== '0' && texto !== '0.00';
                }""", timeout=timeout_datos_ms)
            except PWTimeout:
                print("Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.", file=sys.stderr)
            html_tabla = _extraer_html_tabla(frame)
        finally:
            browser.close()
    if not html_tabla:
        raise RuntimeError("No se pudo extraer la tabla de indicadores.")
    df = pd.read_html(io.StringIO(html_tabla))[0]
    df.columns = [_limpiar_encabezado(columna) for columna in df.columns]
    return df.dropna(axis=1, how="all")

## Normalización y exportación

In [4]:
def _a_snake_case(texto: str) -> str:
    texto = texto.replace("%", "pct")
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "_", sin_acentos).strip("_").lower()


def normalizar_para_analisis(df: pd.DataFrame, momento_extraccion: Optional[datetime] = None) -> pd.DataFrame:
    momento_extraccion = momento_extraccion or datetime.now()
    resultado = df.copy()
    resultado.columns = [_a_snake_case(str(columna)) for columna in resultado.columns]
    for columna in resultado.columns:
        serie = resultado[columna]
        if pd.api.types.is_string_dtype(serie):
            valores = serie.astype(str).str.strip()
            if valores.str.endswith("%").all():
                resultado[columna] = pd.to_numeric(valores.str.rstrip("%"), errors="coerce")
    resultado.insert(0, "fecha_hora_extraccion", momento_extraccion.isoformat(timespec="seconds"))
    resultado.insert(1, "fuente_datos", FUENTE_DATOS)
    return resultado


def exportar_csv_analitico(df: pd.DataFrame, carpeta_salida: Path = CARPETA_SALIDA) -> Path:
    momento = datetime.now()
    carpeta_salida.mkdir(parents=True, exist_ok=True)
    nombre = f"{momento:%Y%m%d_%H%M%S}_indice_fibras_amefibra.csv"
    ruta = carpeta_salida / nombre
    normalizar_para_analisis(df, momento).to_csv(ruta, index=False, encoding="utf-8")
    return ruta

## Ejecutar extracción

In [5]:
import asyncio
import concurrent.futures
import warnings

print(f"Consultando {URL_PAGINA} ...")


def _ejecutar_en_hilo_aparte():
    # El kernel de Jupyter ya corre un event loop de asyncio, y la API síncrona de
    # Playwright no admite ejecutarse dentro de uno (lanza Error), así que se
    # despacha a un hilo aparte. Playwright crea su propio loop internamente con
    # asyncio.new_event_loop(), que en Windows respeta la *policy* activa; el
    # kernel deja configurada una policy basada en SelectorEventLoop (para
    # compatibilidad con zmq/tornado), que no soporta subprocesos, y Playwright
    # necesita subprocesos para lanzar el navegador. Por eso se activa aquí,
    # solo para este hilo, la policy de Proactor que sí los soporta. La API de
    # policies está deprecada desde Python 3.14 (se retira en 3.16) pero sigue
    # siendo, por ahora, el único gancho disponible para influir en qué clase de
    # loop crea Playwright interceptar aquí (Playwright llama a
    # asyncio.new_event_loop() directo, sin exponer alternativa).
    if sys.platform == "win32":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    return obtener_tabla_fibras(headless=HEADLESS, timeout_datos_ms=TIMEOUT_DATOS_MS)


with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    df = executor.submit(_ejecutar_en_hilo_aparte).result()

print(f"Índice FIBRAS - {datetime.now():%Y-%m-%d %H:%M} (dato con ~20 min de retraso)")
display(df)

if EXPORTAR_CSV_ANALITICO:
    ruta_csv_analitico = exportar_csv_analitico(df)
    print(f"CSV analítico guardado en: {ruta_csv_analitico}")

Consultando https://amefibra.com/el-mercado/indice-fibras/ ...


Índice FIBRAS - 2026-08-22 16:23 (dato con ~20 min de retraso)


Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.


,Emisora,Cotización,Var.,Var. %,Apertura,Máx. día,Min. día,Promedio,Operaciones,Volumen,Importe,Máx. 52 s.,Min. 52 s.
0,DANHOS13,28.71,0.10,0.35%,28.77,28.75,29.00,28.54,2262,164212,4717301,29.12,23.65
1,EDUCA18,52.50,-1.50,-2.78%,52.50,52.50,52.50,52.50,11,366,19215,58.36,46.39
2,FIBRAMQ12,43.51,0.76,1.78%,43.05,42.95,43.95,42.16,1536,890429,38525363,45.27,27.73
3,FIBRAPL14,75.57,1.07,1.44%,75.43,74.80,76.06,74.80,4503,513731,38792854,83.97,64.05
4,FIBRAUP18,37.45,0.00,0.00%,37.45,37.45,37.45,37.45,11,32,1194,41.00,17.27
5,FIHO12,7.65,0.11,1.46%,7.59,7.65,7.66,7.53,236,23192,177352,8.01,6.92
6,FINN13,4.86,0.09,1.89%,4.83,4.77,4.88,4.77,179,8250,40058,5.40,4.33
7,FMTY14,14.50,0.28,1.97%,14.43,14.27,14.59,14.26,28161,7460899,107779291,15.79,12.32
8,FNOVA17,41.59,-0.13,-0.31%,42.16,42.01,42.80,41.52,369,13733,572960,45.95,27.00
9,FPLUS16,5.09,0.02,0.39%,5.05,5.10,5.10,5.01,130,10960,55246,6.00,4.82


CSV analítico guardado en: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_162323_indice_fibras_amefibra.csv


In [6]:
if EXPORTAR_CSV_EXCEL:
    df.to_csv(RUTA_CSV_EXCEL, index=False, encoding="utf-8-sig")
    print(f"CSV compatible con Excel guardado en: {RUTA_CSV_EXCEL}")

if EXPORTAR_XLSX:
    df.to_excel(RUTA_XLSX, index=False)
    print(f"Excel guardado en: {RUTA_XLSX}")